# Usage

## Available Cloud Species

The cloud structures are based on the vapour pressures and nucleation rates of the cloud forming materials. The data and their sources are stored in nimbus/data/chem/cloud_material.csv and can be adjusted to your need. If you have requests for further species, contact me (kiefersv.mail@gmail.com). Currently, the following materials are available:

<ul style="columns:4;">
  <li>Al2O3</li>
  <li>C</li>
  <li>CaTiO3</li>
  <li>CH4</li>
  <li>Cr</li>
  <li>CsCl</li>
  <li>Fe</li>
  <li>FeO</li>
  <li>H2O</li>
  <li>H2S</li>
  <li>KCl</li>
  <li>MgSiO3</li>
  <li>Mg2SiO4</li>
  <li>MnS</li>
  <li>NaCl</li>
  <li>Na2S</li>
  <li>NH3</li>
  <li>NH4Cl</li>
  <li>NH4SH</li>
  <li>S2</li>
  <li>S8</li>
  <li>SiO</li>
  <li>SiO2</li>
  <li>TiO2</li>
  <li>ZnS</li>
</ul>


## Main Nimbus functions

Nimbus comes with multiple setting options for the atmospheric structure and the numerical evaluation. Here we go quickly through each function, explaining the different settings available. The main object of Nimbus is defined via the following function which sets general evaluation parameters:

In [ ]:
nimb = Nimbus(working_dir='.', create_analytic_plots=False, verbose=False)

- __working\_dir__: Directory to store all outputs and default search location for inputs.
- __create\_analytic\_plots__: If True, analytic plots of the intermediate outputs are created. This significantly slows down calculation but allows for additional insights and debugging.
- __verbose__: If True, additional information is printed. This affects runtime as a progress bar is enabled.

The atmospheric structure is defined via the following function (which also performs all the structure calculations):


In [ ]:
nimb.set_up_atmosphere(temperature, pressure, kzz, mmw, gravity, species,
                       deep_mmr, fsed=1, metalicity=1)

- __temperature__: Temperature structure of the atmosphere [K]
- __pressure__: Pressure structure of the atmosphere [bar]. Must have same dimension as temperature.
- __kzz__: Mixing strength coefficient of the atmosphere [cm$^2$/s]. Must have same dimension as temperature.
- __mmw__: Mean molecular weight of the atmosphere [amu]. Singular value for the whole atmosphere.
- __gravity__: Gravity of the planet [cm/s$^2$]. Singular value for the whole atmosphere.
- __species__: Name of the cloud material. Only one material can be given.
- __deep\_mmr__: Mass mixing ratio of the cloud below the cloud deck.
- __fsed__: Initial guess of the particle size, given as settling parameter. Default value is typically sufficient.
- __metallicity__: Gas-phase metallicity relative to solar used for the vapour pressure calculations of some cloud materials.

To simulate the cloud structure, the compute function is called. Here, general settings about how the cloud structure should be calculated can be set:

In [ ]:
nimb.compute(typ='convergence', rel_dif_in_mmr=1e-3, max_iterations=None,
             save_file=None)

- __typ__: Determines the type of evaluation. Possibilities are
  - 'convergence': Iterates over fixed cloud particle radii until the maximum difference is below rel_dif_in_mmr.
  - 'iterate': Use a fixed number of iterations (particularly useful for retrival tasks)
  - 'full': Full evaluation of the underlying ODEs (no fixed radius assumption)
- __rel\_dif\_in\_mmr__: If type='convergence', the iteration will stop if the maximum difference is below this value.
- __max\_iterations__: If type='iterate', the iteration will stop after this number of repetitions.
- __save\_file__: If a string is given, the run will be saved under the given name in the working_dir.

## Additional Nimbus Settings

In addition to the main settings, these functions allow for more detailed control over the Nimbus run. In general, it is not expected that you will need to adjust any of these variables. The following function can be used to adjust settings for the ODE solver:

In [ ]:
 nimb.set_solver_settings(
     initial_time_for_solver=None, end_time_for_solver=None,
     evaluation_steps_for_solver=None, degree_of_radius_polinomial=None,
     rtol=None, atol=None, ode_minimum_mmr=None
 )

- __initial\_time\_for\_solver__: This sets the start time of the solver (only relevant for the diagnostic output)
- __end\_time\_for\_solver__: The maximum evaluation time of the solver. It is important to set this valeu large enough for the cloud model to be able to reach a static solution. Default value is 1e15 seconds.
- __evaluation\_steps\_for\_solver__: Number of evaluation points (only relevant for the diagnostic output)
- __degree\_of\_radius\_polinomial__: Degree of the polynomial used for radius smoothing. Default is 8.
- __rtol__: Relative tolarance of the solver. Default is 1e-6.
- __atol__: Absolute tolerance of the solver. Default is 1e-25.
- __ode\_minimum\_mmr__: Lower limit for the solver, everything lower is considered to be 0 (should be lower than atol).

The cloud physics parameters can be set via the following function:

In [ ]:
nimb.set_cloud_settings(minimum_cloud_particle_radius=None, molecular_cross_section=None)

- __minimum\_cloud\_particle\_radius__: Lower size limit of the cloud particles, considered to be the CCN radius. Default is 1e-7 cm.
- __molecular\_cross\_section__: Molecular cross-section. Default is set for an H$_2$ gas and is 2e-15 cm$^2$.

In addition, you might want to add some additional factors to certain parts of the equations. Some of these tweaks (or fudges) are predefined. The default for all values is 1.

In [ ]:
set_fudge_settings(
    nucleation_rate_fudge=None, accreation_rate_fudge=None,
    sticking_coefficient=None
)

- __nucleation\_rate\_fudge__: Multiplicative factor to the nucleation rate to study how uncertainties in the nucleation rate affect the cloud structure.
- __accreation\_rate\_fudge__: Decrypted, same as sticking\_coefficient.
- __sticking\_coefficient__: Chance of a collisional limited accretion reaction to result in the growth of the cloud particle.

## Spectra calculations

To generate spectra from cloud models, Nimbus connects to VIRGA and PICASO. Before a spectrum can be produced, the calculations need to be set up:

In [ ]:
nimb.set_up_spectra_calculation(mass_planet, radius_planet, temperature_star,
                                radius_star, metalicity_star, logg_star,
                                path_to_opacities=None)

- __mass\_planet__: Mass of the planet [M_jup]
- __radius\_planet__: Radius of the planet [R_jup]
- __temperature\_star__: Temperature of the star [K]
- __radius\_star__: Radius of the star [R_sun]
- __metalicity\_star__: Metalicity of the star relative to the sun.
- __logg\_star__: Gravity of the star [log(cm/s$^2$)]
- __path\_to\_opacities__: Nimbus has its own cloud opacities. However, here you can provide another path to your own opacity files (needs to be in the same format as the Nimbus opacity files).

After the cloud structure is set up, the spectrum can be calculated with PICASO. There are multiple intrinsic assumptions within this function and it is advised to adapt this function outside of Nimbus to fit your needs.

In [ ]:
nimb.plot_spectrum(chem_data=None, data=None, typ='transmission', tag='last_run',
                   plot=True, cloud_less=False)